# SuttaLog4 — 빠알리어 음성 생성 (edge-tts)

1. edge-tts 설치
2. tts-texts-new.json 업로드
3. 데바나가리 변환 → MP3 생성
4. manifest.json 병합 → ZIP 다운로드

In [ ]:
!pip install -q edge-tts
print('edge-tts 설치 완료!')

In [ ]:
# 데바나가리 변환 함수
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''; i = 0; s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t\"\'/()': result += ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

# 테스트
print(pali_to_devanagari('Buddhaṃ saraṇaṃ gacchāmi'))
print(pali_to_devanagari('bhikkhave'))
print('변환 함수 준비 완료!')

In [ ]:
# tts-texts-new.json 업로드
from google.colab import files
uploaded = files.upload()
print(f'업로드: {list(uploaded.keys())}')

In [ ]:
# MP3 생성 (edge-tts hi-IN-MadhurNeural)
import edge_tts, os, json, asyncio

VOICE = 'hi-IN-MadhurNeural'
RATE = '-30%'

with open('tts-texts-new.json', 'r', encoding='utf-8') as f:
    texts = json.load(f)
print(f'생성할 텍스트: {len(texts)}개')

os.makedirs('audio_new', exist_ok=True)

async def speak(text, filename):
    comm = edge_tts.Communicate(text, VOICE, rate=RATE)
    await comm.save(filename)

mapping = {}
errors = []

for i, text in enumerate(texts):
    fname = f's4_{i:04d}.mp3'
    outpath = f'audio_new/{fname}'
    mapping[text] = fname
    # 소문자 키도 추가
    lower = text.lower()
    if lower != text:
        mapping[lower] = fname
    if os.path.exists(outpath):
        if (i+1) % 100 == 0: print(f'[{i+1}/{len(texts)}] skip')
        continue
    try:
        clean = text.replace('\n', ' ').strip()
        deva = pali_to_devanagari(clean)
        await speak(deva, outpath)
        if (i+1) % 50 == 0:
            print(f'[{i+1}/{len(texts)}] {clean[:30]}...')
    except Exception as e:
        errors.append((text, str(e)))
        if text in mapping: del mapping[text]
        if lower in mapping: del mapping[lower]
        print(f'[{i+1}] ERR: {e}')

print(f'\n완료! 성공: {len(mapping)}, 오류: {len(errors)}')
if errors:
    print('오류 목록:')
    for t, e in errors[:10]:
        print(f'  {t[:40]}: {e}')

In [ ]:
# manifest.json 저장 + ZIP 생성
import zipfile
from google.colab import files

# 새 매니페스트 저장
with open('audio_new/manifest-new.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print(f'manifest-new.json: {len(mapping)}개 엔트리')

# ZIP 압축
with zipfile.ZipFile('suttalog4-audio-new.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir('audio_new'):
        zf.write(f'audio_new/{fn}', fn)

size = os.path.getsize('suttalog4-audio-new.zip') / (1024*1024)
print(f'suttalog4-audio-new.zip ({size:.1f} MB)')
files.download('suttalog4-audio-new.zip')
print('다운로드 시작!')

## 다운로드 후 작업

1. ZIP 해제
2. MP3 파일들을 `public/audio/`에 복사
3. `manifest-new.json`을 기존 `manifest.json`에 병합:

```bash
# Node.js로 manifest 병합
node -e "
const fs = require('fs');
const old = JSON.parse(fs.readFileSync('public/audio/manifest.json'));
const add = JSON.parse(fs.readFileSync('manifest-new.json'));
const merged = { ...old, ...add };
fs.writeFileSync('public/audio/manifest.json', JSON.stringify(merged, null, 2));
console.log('병합 완료:', Object.keys(merged).length, '개');
"
```